In [ ]:
# ==============================================================================
# Phase 2: AI Categorization & Human Audit Workflow
# 
# DESCRIPTION:
# Enforces a tripartite Source of Truth: Master (Metadata), CSV (AI), Excel (Human).
# It compares hashes to detect changes, queries the LLM for updates, appends 
# those to a persistent CSV log, and builds a fresh Excel workbench by safely 
# overlaying human audits onto uncorrupted master data.
# ==============================================================================
# 
# PHASE 2: AI CATEGORIZATION & HUMAN AUDIT WORKFLOW (CDC AWARE)
# 
# DESCRIPTION:
# This script executes the Change Data Capture (CDC) categorization pipeline. 
# It uses a Tripartite Architecture to protect your manual labor while keeping 
# metadata synchronized across three distinct sources of truth:
#   1. Master Dataset: The absolute truth for upstream metadata (Labels, Paths).
#   2. AI Log (CSV): An append-only ledger of Gemini LLM predictions.
#   3. Human Workbench (Excel): The absolute truth for sociological classifications.
#
# ------------------------------------------------------------------------------
# STANDARD OPERATING PROCEDURE:
# 1. RUN SCRIPT: The script mathematically compares cryptographic hashes across 
#    the three files, sends net-new or un-reviewed concepts to the Gemini API, 
#    and safely rebuilds 'data/interim/human_category_review.xlsx'.
# 2. AUDIT IN EXCEL: Open 'human_category_review.xlsx' and sort by 'audit_trigger'.
# 3. ASSIGN CATEGORY: Select the correct category in the 'category_human' column.
# 4. CLEAR TRIGGER [CRITICAL]: Once satisfied with a row, DELETE the text in the 
#    'audit_trigger' column. This signals the pipeline that the audit is complete.
# 5. SAVE & CLOSE: Save the Excel file. Proceed to notebook 03_build_application_dataset.ipynb. 
#
# ------------------------------------------------------------------------------
# HANDLING SPECIFIC SCENARIOS:
# 
# [A] NEW UPSTREAM DATA INGESTED: 
#     The script detects new hashes, sends the concepts to Gemini, appends them 
#     to the AI log, and inserts them into Excel with the trigger "New Concept".
#
# [B] EXISTING METADATA UPDATED (e.g., Fixing a typo):
#     The script detects the mutated hash. 
#       - If already categorized: The "Human Override Lock" engages. It BYPASSES 
#         the API to save compute and flags the row as "Metadata Updated" in Excel. 
#         Verify the typo fix didn't change the meaning, then clear the trigger.
#       - To force an AI re-evaluation: Delete your category from 'category_human' 
#         in Excel, then run this script again.
#
# [C] UNWANTED CONCEPTS DELETED IN UPSTREAM RAW DATA:
#     If a concept is deleted from the raw upstream files and 01_consolidate_master_dataset.ipynb is run, 
#     the script will identify the "ghosts" and automatically purge  
#     them from your Excel workbench when this script is run. No manual Excel cleanup is required.
#
# [D] API FAILED OR DROPPED A CONCEPT:
#     If a network batch fails or the LLM silently drops a row, the script flags 
#     it internally as "Failed". You do not need to do anything; the script will 
#     AUTOMATICALLY RETRY sending that concept to the API on the next run.
#
# ------------------------------------------------------------------------------
# [!] DATA INTEGRITY WARNING: 
# Do not manually edit the CURIE, Primary_Label, or Hierarchy_Path directly inside 
# the Excel workbench. Excel is strictly a tracking mechanism for the category_human 
# and audit_trigger columns. Metadata edits made in Excel will be ignored 
# and overwritten by the Master dataset on the next run. Fix upstream data only.
# ==============================================================================

import os
import sys
import json
import time
import pandas as pd
import xlsxwriter
from xlsxwriter.utility import xl_col_to_name
from google import genai
from google.genai import types
from tqdm import tqdm
from dotenv import load_dotenv

# --- 1. SETUP & DIRECTORIES ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
interim_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

load_dotenv(os.path.join(config_dir, ".env"))
try:
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
except Exception as e:
    print(f"[!] API Initialization Error: {e}")
    sys.exit(1)

master_file = os.path.join(interim_dir, "master_ontology_dataset.csv")
excel_file = os.path.join(interim_dir, "human_category_review.xlsx")
mapping_file = os.path.join(interim_dir, "category_mapping_ai.csv") 
category_config_file = os.path.join(config_dir, "categories.json")

# --- 2. LOAD CATEGORY CONFIGURATION ---
try:
    with open(category_config_file, "r") as f:
        CATEGORY_CONFIG = json.load(f)
except FileNotFoundError:
    print(f"[!] CRITICAL: Could not find {category_config_file}")
    sys.exit(1)

ALLOWED_LABELS = [v['label'] for v in CATEGORY_CONFIG.values()]
MODEL_NAME = 'gemini-2.5-flash'

response_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "curie": {"type": "string"},
            "category": {"type": "string", "enum": ALLOWED_LABELS},
            "confidence": {"type": "number"}
        },
        "required": ["curie", "category", "confidence"]
    }
}

SYSTEM_INSTRUCTION = f"""
You are an expert ontologist specializing in the scientific study of religion. Your task is to categorize the 'Subject' concept.

DATA SCHEMA GUIDANCE:
- Subject (Primary_Label): This is the core concept to be categorized.
- supporting evidence: Use Synonyms, Hierarchy_Path, Source_System, and Description to clarify the meaning of the Subject.
- objective: Determine which category best fits the SUBJECT.

STRICT CATEGORY DEFINITIONS:
{json.dumps(CATEGORY_CONFIG, indent=2)}

STRICT DECISION RULES:
1. COMMUNITIES vs IDENTITIES:
   - Use 'communities' ONLY for GENERIC COMMON NOUNS representing types of social units (e.g., 'diocese', 'parish').
   - If the Subject is a PROPER NAME of a group (e.g., 'First Baptist Church'), use 'identities' or 'religious other'.
"""

# --- 3. LOAD TRIPARTITE DATA & GRACEFUL LEGACY MIGRATION ---
print("[*] Loading Master Dataset (Source of Truth for Metadata)...")
if not os.path.exists(master_file):
    print(f"[!] CRITICAL: {master_file} not found. Run Script 1 first.")
    sys.exit(1)
    
master_df = pd.read_csv(master_file, dtype={'CURIE': str})
master_df = master_df.rename(columns={'row_hash': 'master_hash'})

print("[*] Loading AI Log (Source of Truth for Predictions)...")
if os.path.exists(mapping_file):
    # Load without enforcing ai_hash dtype initially to prevent ValueError on legacy files
    ai_df = pd.read_csv(mapping_file, dtype={'CURIE': str})
    
    # LEGACY UPGRADE: Backfill missing ai_hash
    if 'ai_hash' not in ai_df.columns:
        print("    [+] Upgrading legacy AI Log to V2 CDC architecture...")
        ai_df = ai_df.merge(master_df[['CURIE', 'master_hash']], on='CURIE', how='left')
        ai_df.rename(columns={'master_hash': 'ai_hash'}, inplace=True)
        # Save the upgraded file immediately
        ai_df.to_csv(mapping_file, index=False, encoding='utf-8-sig')
        
    ai_df = ai_df.drop_duplicates(subset=['CURIE'], keep='last') 
else:
    ai_df = pd.DataFrame(columns=['CURIE', 'category', 'confidence', 'ai_hash', 'processing_status'])

print("[*] Loading Excel Workbench (Source of Truth for Human Audits)...")
if os.path.exists(excel_file):
    raw_excel = pd.read_excel(excel_file, dtype={'CURIE': str})
    
    # LEGACY UPGRADE: Backfill missing last_audited_hash and audit_trigger
    if 'last_audited_hash' not in raw_excel.columns:
        print("    [+] Upgrading legacy Excel Workbench to V2 CDC architecture...")
        # If they already audited it (has a category), assume it matches the current master hash
        raw_excel = raw_excel.merge(master_df[['CURIE', 'master_hash']], on='CURIE', how='left')
        raw_excel['last_audited_hash'] = raw_excel.apply(
            lambda r: r['master_hash'] if pd.notna(r.get('category_human')) else None, axis=1
        )
        raw_excel['audit_trigger'] = "Unchanged" # Default legacy rows to unchanged
        
    expected_cols = ['CURIE', 'category_human', 'last_audited_hash', 'audit_trigger']
    for col in expected_cols:
        if col not in raw_excel.columns:
            raw_excel[col] = None
            
    excel_df = raw_excel[expected_cols].copy()
else:
    excel_df = pd.DataFrame(columns=['CURIE', 'category_human', 'last_audited_hash', 'audit_trigger'])

# --- 4. DETERMINISTIC ID RECOVERY ---
excel_curies = set(excel_df['CURIE'].dropna())
master_curies = set(master_df['CURIE'].dropna())

dropped_curies = excel_curies - master_curies
new_curies = master_curies - excel_curies

if dropped_curies and new_curies and not excel_df.empty:
    dropped_df = excel_df[excel_df['CURIE'].isin(dropped_curies)].merge(raw_excel[['CURIE', 'Primary_Label', 'Source_System']], on='CURIE')
    new_master_df = master_df[master_df['CURIE'].isin(new_curies)]
    
    recovery_merge = pd.merge(
        new_master_df[['CURIE', 'Primary_Label', 'Source_System']],
        dropped_df[['Primary_Label', 'Source_System', 'category_human', 'last_audited_hash', 'audit_trigger']],
        on=['Primary_Label', 'Source_System'],
        how='inner'
    )
    if not recovery_merge.empty:
        print(f"[*] Recovered human audits for {len(recovery_merge)} mutated IDs.")
        excel_df = pd.concat([excel_df, recovery_merge], ignore_index=True)
        # Prune the ghost records that were successfully ported
        excel_df = excel_df[~excel_df['CURIE'].isin(dropped_curies)]

# --- 5. PREPARE AI QUEUE ---
print("[*] Evaluating concepts for AI Categorization...")

# Join master to AI log to see what the AI has processed vs what it needs to process
api_prep_df = master_df.merge(ai_df[['CURIE', 'category', 'confidence', 'ai_hash', 'processing_status']], on='CURIE', how='left')

# Bring in the human state to prevent redundant AI calls on already-audited data
api_prep_df = api_prep_df.merge(excel_df[['CURIE', 'category_human']], on='CURIE', how='left')

# HUMAN OVERRIDE LOCK: Only send to Gemini if it failed previously, OR if the hash mutated
# AND the human hasn't already locked in a decision.
is_failed = api_prep_df['processing_status'] != 'Success'
hash_mutated = api_prep_df['master_hash'] != api_prep_df['ai_hash']
needs_human = api_prep_df['category_human'].isna() | (api_prep_df['category_human'].astype(str).str.strip() == "")

api_queue = api_prep_df[is_failed | (hash_mutated & needs_human)].copy()

# --- 6. API EXECUTION LOOP ---
if api_queue.empty:
    print("[*] All hashes match. No net-new or updated concepts for the AI.")
else:
    print(f"[*] Queued {len(api_queue)} concepts for Gemini processing...")

    def process_batch(chunk):
        prompt_items = []
        for _, r in chunk.iterrows():
            syns = str(r['Synonyms']) if pd.notna(r['Synonyms']) else "None"
            desc = str(r['Description'])[:150] if pd.notna(r['Description']) else "None"
            prompt_items.append(
                f"ID: {r['CURIE']} | Subject: {r['Primary_Label']} | Synonyms: {syns} | "
                f"Path: {r['Hierarchy_Path']} | Source: {r['Source_System']} | Desc: {desc}"
            )
        prompt_text = "Classify this batch of religious concepts:\n" + "\n".join(prompt_items)
        
        try:
            response = client.models.generate_content(
                model=MODEL_NAME, contents=prompt_text,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    response_mime_type="application/json",
                    response_schema=response_schema
                )
            )
            return response.parsed, "Success"
        except Exception as e:
            print(f"\n[!] API Batch Error: {e}")
            return None, "Failed"

    BATCH_SIZE = 50 
    
    for i in tqdm(range(0, len(api_queue), BATCH_SIZE)):
        chunk = api_queue.iloc[i:i+BATCH_SIZE]
        results, status = process_batch(chunk)
        
        batch_results = []
        if status == "Success" and results:
            results_df = pd.DataFrame(results)
            results_df['curie'] = results_df['curie'].astype(str)
            
            # Drop stale columns to prevent Pandas from creating category_x and category_y
            clean_chunk = chunk.drop(columns=['category', 'confidence', 'processing_status', 'ai_hash'], errors='ignore')
            merged = clean_chunk.merge(results_df, left_on='CURIE', right_on='curie', how='left')
            
            for _, row in merged.iterrows():
                # VALIDATION: Detect if the LLM silently dropped this concept
                is_missing = pd.isna(row.get('category')) or str(row.get('category')).strip() in ["", "nan", "None"]
                
                batch_results.append({
                    'CURIE': row['CURIE'],
                    'Primary_Label': row['Primary_Label'],
                    'category': None if is_missing else row['category'],
                    'confidence': 0 if is_missing else row['confidence'],
                    'Source_System': row['Source_System'],
                    'Hierarchy_Path': row['Hierarchy_Path'],
                    'processing_status': 'Failed' if is_missing else 'Success',
                    'ai_hash': None if is_missing else row['master_hash']
                })
        else:
            for _, row in chunk.iterrows():
                # STRICT SCHEMA ENFORCEMENT FOR AI CACHE (FAILURES)
                batch_results.append({
                    'CURIE': row['CURIE'], 
                    'Primary_Label': row['Primary_Label'],
                    'category': None, 
                    'confidence': 0, 
                    'Source_System': row['Source_System'],
                    'Hierarchy_Path': row['Hierarchy_Path'],
                    'processing_status': 'Failed',
                    'ai_hash': None 
                })

        # APPEND TO INSPECTABLE CSV LOG
        batch_output_df = pd.DataFrame(batch_results)
        
        # Enforce column order to ensure it remains rectangular
        csv_cols = ['CURIE', 'Primary_Label', 'category', 'confidence', 'Source_System', 'Hierarchy_Path', 'processing_status', 'ai_hash']
        batch_output_df = batch_output_df[csv_cols]
        
        file_exists = os.path.isfile(mapping_file)
        batch_output_df.to_csv(mapping_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
        time.sleep(1.2)
        
    # Reload the AI df now that the API is done to get the freshest data for the Excel build
    ai_df = pd.read_csv(mapping_file, dtype={'CURIE': str, 'ai_hash': str}).drop_duplicates(subset=['CURIE'], keep='last')

# --- 7. EXCEL SYNCHRONIZATION & FORMATTING ---
print(f"[*] Assembling strict master merge for {os.path.basename(excel_file)}...")

# Join the Master (Metadata) + AI CSV (Predictions) + Excel (Human Audits)
working_df = master_df.merge(
    ai_df[['CURIE', 'category', 'confidence', 'processing_status']], on='CURIE', how='left'
).merge(
    excel_df[['CURIE', 'category_human', 'last_audited_hash', 'audit_trigger']], on='CURIE', how='left'
)

# RECONCILIATION: Sync the tracking hash if the human has audited the row.

# Condition 1: The human explicitly deleted the text in the audit_trigger column.
# We must ensure it has an existing hash. A brand-new concept will have a NaN 
# trigger simply because it didn't exist in Excel yet, not because a human cleared it.
is_blank_trigger = working_df['audit_trigger'].isna() | (working_df['audit_trigger'].astype(str).str.strip() == "")
has_existing_hash = working_df['last_audited_hash'].notna()
cleared_mask = is_blank_trigger & has_existing_hash

# Condition 2: The human populated a category for a previously un-audited "New Concept".
has_category = working_df['category_human'].notna() & (working_df['category_human'].astype(str).str.strip().str.lower() != 'nan') & (working_df['category_human'].astype(str).str.strip() != "")
was_new_concept = working_df['last_audited_hash'].isna()

# Execute sync
working_df.loc[cleared_mask | (has_category & was_new_concept), 'last_audited_hash'] = working_df['master_hash']

# COMPUTE AUDIT TRIGGERS FOR THE USER
def determine_trigger(row):
    if pd.isna(row['last_audited_hash']):
        return "New Concept"
    elif row['master_hash'] != row['last_audited_hash']:
        return "Metadata Updated"
    else:
        return "Unchanged"

working_df['audit_trigger'] = working_df.apply(determine_trigger, axis=1)

# Format for Export
export_cols = [
    'CURIE', 'Primary_Label', 'Source_System', 'Hierarchy_Path', 
    'category', 'confidence', 'category_human', 'audit_trigger', 
    'last_audited_hash', 'processing_status'
]
export_df = working_df[export_cols].copy()

# Sort actionable items to the top
export_df['trigger_rank'] = export_df['audit_trigger'].map({'Metadata Updated': 1, 'New Concept': 2, 'Unchanged': 3})
export_df = export_df.sort_values(by=['trigger_rank', 'Source_System', 'Hierarchy_Path']).drop(columns=['trigger_rank'])

print(f"[*] Safely overwriting Excel workbench...")
writer = pd.ExcelWriter(excel_file, engine='xlsxwriter')
export_df.to_excel(writer, index=False, sheet_name='Audit')

workbook = writer.book
worksheet = writer.sheets['Audit']

text_format = workbook.add_format({'num_format': '@'})
decimal_format = workbook.add_format({'num_format': '0.00'})

curie_idx = export_df.columns.get_loc('CURIE')
worksheet.set_column(curie_idx, curie_idx, 20, text_format) 

conf_idx = export_df.columns.get_loc('confidence')
worksheet.set_column(conf_idx, conf_idx, 12, decimal_format)

worksheet.set_column(export_df.columns.get_loc('Primary_Label'), export_df.columns.get_loc('Hierarchy_Path'), 35)

cat_human_idx = export_df.columns.get_loc('category_human')
col_letter = xl_col_to_name(cat_human_idx)
worksheet.data_validation(f'{col_letter}2:{col_letter}1048576', {
    'validate': 'list',
    'source': ALLOWED_LABELS
})

writer.close()
print(f"[COMPLETE] Run finished. Open '{os.path.basename(excel_file)}' to review updates.")

In [ ]:
# ==============================================================================
# STEP 2.1: INTEGRITY & STATE DIAGNOSTIC (CDC AWARE)
# Evaluates the Excel workbench for completeness and provides a summary of 
# the Change Data Capture (CDC) queue.
# ==============================================================================
import os
import sys
import pandas as pd

print("\n" + "="*60)
print(" PIPELINE INTEGRITY & STATE DIAGNOSTIC ")
print("="*60)

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

master_file = os.path.join(interim_dir, "master_ontology_dataset.csv")
excel_file = os.path.join(interim_dir, "human_category_review.xlsx")
mapping_file = os.path.join(interim_dir, "category_mapping_ai.csv")

try:
    master_df = pd.read_csv(master_file, dtype={'CURIE': str})
    excel_df = pd.read_excel(excel_file, dtype={'CURIE': str})
    
    if os.path.exists(mapping_file):
        ai_df = pd.read_csv(mapping_file, dtype={'CURIE': str})
        ai_df = ai_df.drop_duplicates(subset=['CURIE'], keep='last')
    else:
        ai_df = pd.DataFrame(columns=['CURIE'])
        
except FileNotFoundError as e:
    print(f"[!] Error loading files: {e}")
    sys.exit(1)

print(f"Row Counts -> Master: {len(master_df)} | Excel Workbench: {len(excel_df)} | AI Log Entries: {len(ai_df)}\n")

# --- A. WORKBENCH COMPLETENESS ---
print("--- A. WORKBENCH STRUCTURAL COMPLETENESS ---")
master_curies = set(master_df['CURIE'].dropna())
excel_curies = set(excel_df['CURIE'].dropna())

missing_from_excel = master_curies - excel_curies
ghosts_in_excel = excel_curies - master_curies

print(f"Missing from Excel : {len(missing_from_excel)} (Needs to be synced by Script 2)")
print(f"Ghosts in Excel    : {len(ghosts_in_excel)} (Orphaned IDs that need purging)")
print("")

# --- B. CDC QUEUE STATE ---
print("--- B. CHANGE DATA CAPTURE (CDC) STATE ---")
if 'audit_trigger' in excel_df.columns:
    # Fill NAs to handle blank strings cleanly
    excel_df['audit_trigger'] = excel_df['audit_trigger'].fillna('Unchanged')
    
    # Calculate queue sizes
    pending_human = len(excel_df[excel_df['audit_trigger'] != 'Unchanged'])
    unchanged = len(excel_df[excel_df['audit_trigger'] == 'Unchanged'])
    
    print(f"Pending Human Audit : {pending_human} concepts")
    print(f"Fully Finalized     : {unchanged} concepts")
    
    if pending_human > 0:
        print("\n  -> Breakdown of pending work:")
        trigger_counts = excel_df[excel_df['audit_trigger'] != 'Unchanged']['audit_trigger'].value_counts()
        for trigger, count in trigger_counts.items():
            print(f"     - {trigger}: {count}")
else:
    print("[!] 'audit_trigger' column missing. Run Script 2 to initialize CDC tracking.")

print("\n" + "="*60)
print(" DIAGNOSTIC COMPLETE ")
print("="*60)

In [ ]:
# ==============================================================================
# STEP 2.2: AI PERFORMANCE AUDIT REPORTING (CDC AWARE)
# 
# DESCRIPTION:
# Generates performance metrics to document the reliability of the zero-shot LLM.
# It explicitly filters out any concepts pending human review to ensure it 
# only grades the AI against finalized, audited decisions.
# ==============================================================================

import os
import pandas as pd

print("\n" + "="*80)
print(" STEP 2.2: GENERATING AI PERFORMANCE REPORTS ")
print("="*80)

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

human_file = os.path.join(interim_data_dir, "human_category_review.xlsx")
report_source_file = os.path.join(interim_data_dir, "report_changes_by_source.csv")
report_category_file = os.path.join(interim_data_dir, "report_changes_by_category.csv")

print(f"[*] Loading audited dataset from: {os.path.basename(human_file)}...")
try:
    df = pd.read_excel(human_file)
except FileNotFoundError:
    print(f"[!] ERROR: Could not find {human_file}.")
    exit(1)

# --- FILTER PENDING REVIEWS ---
# Only evaluate rows that are explicitly finalized (i.e., NOT marked as needing review)
active_triggers = ['New Concept', 'Metadata Updated']
pending_mask = df['audit_trigger'].astype(str).str.strip().isin(active_triggers)

pending_count = pending_mask.sum()
reviewed_df = df[~pending_mask].copy()

print(f"[*] Total concepts in workbench : {len(df)}")
print(f"[*] Concepts pending audit    : {pending_count} (Excluded from metrics)")
print(f"[*] Concepts fully finalized  : {len(reviewed_df)}")

if reviewed_df.empty:
    print("\n[!] No finalized concepts available to grade. Complete your audit first.")
else:
    # Standardize text
    reviewed_df['category'] = reviewed_df['category'].fillna('').astype(str).str.strip().str.lower()
    reviewed_df['category_human'] = reviewed_df['category_human'].fillna('').astype(str).str.strip().str.lower()

    # Create boolean flag for overrides
    reviewed_df['was_changed'] = reviewed_df['category'] != reviewed_df['category_human']

    # --- TABLE 1: PERFORMANCE BY SOURCE SYSTEM ---
    source_summary = reviewed_df.groupby('Source_System').agg(
        Total_Concepts=('CURIE', 'count'),
        Unchanged=('was_changed', lambda x: (~x).sum()),
        Changed=('was_changed', 'sum')
    ).reset_index()

    source_summary['%_Changed'] = (source_summary['Changed'] / source_summary['Total_Concepts'] * 100).round(1)
    source_summary = source_summary.sort_values(by='%_Changed', ascending=False)

    # --- TABLE 2: PERFORMANCE BY ORIGINAL AI CATEGORY ---
    cat_summary = reviewed_df.groupby('category').agg(
        Total_Assigned_By_AI=('CURIE', 'count'),
        Unchanged=('was_changed', lambda x: (~x).sum()),
        Changed=('was_changed', 'sum')
    ).reset_index()

    cat_summary['%_Changed'] = (cat_summary['Changed'] / cat_summary['Total_Assigned_By_AI'] * 100).round(1)
    cat_summary = cat_summary.sort_values(by='%_Changed', ascending=False)

    # --- DISPLAY AND EXPORT ---
    print("\n" + "="*80)
    print(" TABLE 1: AI CATEGORIZATION PERFORMANCE BY SOURCE SYSTEM")
    print("="*80)
    print(source_summary.to_string(index=False))

    print("\n" + "="*80)
    print(" TABLE 2: AI CATEGORIZATION PERFORMANCE BY ORIGINAL CATEGORY")
    print("="*80)
    print(cat_summary.to_string(index=False))

    source_summary.to_csv(report_source_file, index=False)
    cat_summary.to_csv(report_category_file, index=False)

    print("\n[*] Reports successfully generated and saved to 'data/interim/'")